# Notebook 01: Data Exploration & Preprocessing (Stage 1 & Stage 2)

This notebook demonstrates loading social media post data, schema verification, and sequential preprocessing stages:
- **Stage 1**: Schema Ingestion, Field Mapping & Pydantic Validation (`DataCollector`)
- **Stage 2.1**: Deduplication (`remove_duplicates`)
- **Stage 2.2**: Spam & Bot Filtering (`filter_spam_and_bots`)
- **Stage 2.3**: Text Cleaning (HTML, whitespace, URLs)
- **Stage 2.4**: Text Normalization (Unicode NFKC, lowercase, repeated characters)
- **Stage 2.5**: Preserve Semantic Information (Hashtags, Emojis, Negation context)
- **Stage 2.6**: End-to-End `PreprocessingPipeline` Execution & Output Verification

### 1. Imports & Configuration Setup

In [ ]:
import sys
import os
sys.path.append('..')

# Ensure UTF-8 output encoding on Windows consoles
if hasattr(sys.stdout, 'reconfigure'):
    try:
        sys.stdout.reconfigure(encoding='utf-8')
    except Exception:
        pass

import pandas as pd
import numpy as np
from loguru import logger

# Stage 01 imports
from src.stage_01_data_collection import (
    DataCollector,
    RawPostSchema,
    SocialDataBatch
)
from src.stage_01_data_collection.collector import standardize_columns

# Stage 02 imports
from src.stage_02_preprocessing import (
    PreprocessingPipeline,
    remove_duplicates,
    filter_spam_and_bots,
    clean_text,
    normalize_text,
    preserve_semantic_information
)

# File I/O utilities
from src.utils.file_io import load_yaml, load_dataframe, save_dataframe

config = load_yaml('../configs/config.yaml')
print('All modules and configurations successfully imported.')

### 2. Inspect Raw Social Media Dataset
Check available raw datasets in `data/01_raw/`.

In [ ]:
raw_csv_path = '../data/01_raw/social_listening_30days.csv'
df_raw = pd.read_csv(raw_csv_path)
print(f'Raw dataset shape: {df_raw.shape[0]} rows, {df_raw.shape[1]} columns')
df_raw.head(3)

### 3. Stage 1: Data Collection, Ingestion & Pydantic Validation
Standardize schema column aliases and validate via `RawPostSchema`.

In [ ]:
collector = DataCollector(output_path='../data/01_raw/raw_social_posts.parquet')
df_validated = collector.collect_from_file(raw_csv_path)
print(f'Successfully validated & ingested {len(df_validated)} posts.')
df_validated.info()

### 4. Stage 2: Step-by-Step Preprocessing Exploration
#### 4.1 Remove Duplicates

In [ ]:
df_dedup = remove_duplicates(df_validated, subset=['text'])
print(f'Before: {len(df_validated)} | After deduplication: {len(df_dedup)}')
df_dedup.head(2)

#### 4.2 Spam & Bot Filtering
Filters based on follower thresholds, excessive hashtags, and URL density.

In [ ]:
df_filtered = filter_spam_and_bots(df_dedup, min_followers=5, max_hashtags=15, max_urls=4)
print(f'Remaining authentic posts: {len(df_filtered)}')
df_filtered.head(2)

#### 4.3 Text Cleaning, Normalization & Semantic Preservation Demonstration

In [ ]:
sample_texts = [
    'Check this out!!! <a href="http://example.com">LINK</a> #AI #Tech https://bit.ly/3xyz 🔥🔥',
    'I CANNOT believe this new model... sooo goooood!! 🚀',
    'Standard text normalization test with mixed Việt ngữ & Unicode café/café.'
]

for raw_text in sample_texts:
    cleaned = clean_text(raw_text, remove_url=True)
    normalized = normalize_text(cleaned, form='NFKC', lowercase=True, max_repeats=2)
    preserved = preserve_semantic_information(normalized, demojize=False)
    
    print(f'RAW:        {raw_text}')
    print(f'PROCESSED:  {preserved}')
    print('-' * 70)

### 5. Execute Full Preprocessing Pipeline & Save Outputs
Runs the complete Stage 2 pipeline on the validated dataset and saves to `data/02_preprocessed/preprocessed_posts.parquet`.

In [ ]:
pipeline = PreprocessingPipeline(config.get('stage_02_preprocessing', {}))
pipeline.output_path = '../data/02_preprocessed/preprocessed_posts.parquet'
df_preprocessed = pipeline.run(df_validated)

print(f'Preprocessing successfully completed! Final dataset shape: {df_preprocessed.shape}')
df_preprocessed[['post_id', 'text', 'cleaned_text', 'likes', 'shares', 'comments']].head(5)